# High-Recall Entity Mention Extraction

Extracts every entity mention verbatim — optimised for **recall** over precision.  
Uses `gpt-4.1` with structured JSON-schema output.  
Each occurrence is listed separately (no deduplication here — that is a downstream step).  
Returns `text`, `type`, and `context` per mention.

Why recall-first?
- A missed mention is unrecoverable once the extraction pass is done.
- Spurious mentions are cheap to filter later (embedding similarity, type checks, etc.).

In [35]:
import json
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from env

## Extraction function

The system prompt instructs the model to:
- include every mention, even uncertain ones (`recall > precision`)
- copy `text` and `context` **verbatim** from the source
- assign a coarse type label (wrong labels are acceptable, missed mentions are not)

In [37]:
SYSTEM_PROMPT = """You are an entity mention extractor for knowledge-graph construction. This is Stage 1 of a pipeline: each mention you emit is later embedded (exact span + surrounding sentence) and resolved into a graph node. You extract entity ENDPOINTS only — never relations, never scalar values.

TASK
Extract every span that names or specifically refers to an entity. List each occurrence separately, even when two mentions refer to the same thing. Never merge, normalize, or deduplicate — resolution is a downstream stage.

WHAT COUNTS AS A MENTION
1. Proper names: people, organizations, places, facilities, products, software, systems, events, works (papers, books, reports, standards), datasets. Numerals inside names are part of the name ("HippoRAG 2", "GPT-4", "World War II").
2. Named technical terms: terms that function as the established name of a concept, method, or artifact in the domain ("entity resolution", "personalized PageRank", "contrastive loss") — as opposed to ad-hoc descriptions.
3. Specific-referent nominals: a definite or demonstrative noun phrase ("the company", "this model", "our pipeline") counts ONLY if a reader of this document alone could identify the unique entity it refers to (it has an in-document antecedent or a unique referent). If it denotes a kind or class, exclude it.

WHAT IS NEVER A MENTION
- Pronouns and bare demonstratives: "it", "they", "he", "this"/"that" without a noun head, relative pronouns.
- Generic common nouns and bare plurals ("users", "data", "the results", "systems", "an approach"), even domain-flavored ones, unless they are part of a longer specific term.
- Scalar values and attributes: numbers, money, counts, dates, times, percentages, durations, measurements. A separate stage handles these. ("2023" is never a mention; "GPT-4" is — the numeral is part of the name.)
- Relations or facts of any kind.

SPAN BOUNDARIES
- Extract the maximal span that functions as a single name or term. Never extract a proper-name substring of a longer proper name: "University of Toronto" yields exactly one mention, not also "Toronto".
- DO separately extract a name that makes its own referential contribution inside a larger phrase — as a modifier of a common-noun head, in a possessive, or in coordination: "the Neo4j database" yields "Neo4j"; "the universities of Toronto and Waterloo" yields "Toronto" and "Waterloo".
- Acronym introduced alongside its expansion, e.g. "Personalized PageRank (PPR)": emit TWO mentions — "Personalized PageRank" and "PPR" — sharing the same context. Do not emit the combined parenthesized span. Extract later standalone occurrences of the acronym normally, one per occurrence.
- Exclude leading articles from proper names ("the" in "the University of Toronto" is not part of the mention) but keep the determiner when the mention is a nominal ("the company", "this model") — it carries the definiteness signal.

VERBATIM REQUIREMENTS
- "text": copied character-for-character from the source, original casing and punctuation intact. It must be findable by exact string search inside its "context".
- "context": the complete sentence containing the mention, copied verbatim. If the mention appears in a heading, list item, table cell, or other fragment without sentence punctuation, use the entire line or cell. If the mention spans a sentence boundary, include every sentence it touches. Never paraphrase, trim, or repair the source text.

TYPE FIELD
Use a lowercase singular noun. Prefer one of these anchor types whenever it fits:
person, organization, location, facility, product, software, method, concept, event, work, dataset, field
Tie-breaks: a runnable code artifact, library, or system → software (over product, over tool); a procedure, algorithm, or technique → method; an abstract idea that is not a procedure → concept; a paper, book, report, or standard → work; an academic or industry discipline → field. Mint a new type only when no anchor fits, and make it a single generic lowercase noun that would be reused verbatim in unrelated documents ("gene", "chemical", "regulation") — never a document-specific compound.

POSTURE
Favor recall on names and established terms — a missed mention is unrecoverable downstream, a spurious one is filtered later. The single exception: generic nominals and class-level terms, where precision wins, because unresolvable generic nodes corrupt graph traversal. When unsure whether a span is an established term or a mere description, ask: is this a name someone would query the graph for, or a phrase describing something? Names and terms in; descriptions out."""

MENTION_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "mentions": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "text":    {"type": "string"},
                    "type":    {"type": "string"},
                    "context": {"type": "string"},
                },
                "required": ["text", "type", "context"],
            },
        }
    },
    "required": ["mentions"],
}


def extract_mentions(text: str) -> list[dict]:
    """Extract entity mentions from a single text chunk with high recall."""
    response = client.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "entity_mentions",
                "strict": True,
                "schema": MENTION_SCHEMA,
            },
        },
    )
    result = json.loads(response.choices[0].message.content)
    return result["mentions"]

## Document chunking

Long documents are split into ~1000-char chunks before extraction so each fits comfortably in context.

In [41]:
def chunk_doc(text: str, max_chars: int = 1000) -> list[str]:
    """Split by double newline, merge small paragraphs into ~max_chars chunks."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) <= max_chars:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks


def extract_mentions_from_doc(doc: str) -> list[dict]:
    """Chunk a document and extract entity mentions; each mention carries episode metadata."""
    all_mentions: list[dict] = []
    for episode_idx, episode in enumerate(chunk_doc(doc)):
        episode_mentions = extract_mentions(episode)
        for m in episode_mentions:
            m["episode_idx"] = episode_idx
            m["episode"]     = episode
        all_mentions.extend(episode_mentions)
    return all_mentions

## Demo

In [74]:
doc = """based on [[Breaking point for Buyers of AI KMS]] and [[Potential Markets]] clearly I need to reach out to 
## **Reach out to CTO's or COO's of a late Seed or Series A B2B SaaS startups**

- ==Late Seed / Series A startups==
	- raised seed couple months ago to feel the pain
	- Fintech, biotech, B2B SaaS
	- **Champions + Buyers**
		- **CTO** / VP of Engineering ==best at seed==
			- answers lots of questions
			- knows KM and its problems
			- owns the tools budget
			- under pressure from stakeholders to reduce per-person risk (KM problme )
		- **COO** / Head of Ops <-- LITERALLY OWNS THE PROBLEM ==best at Ser A==
			- **their job is to run company without founders touching everything**
			- slow onboarding, repeated questions, tribal knowledge <- bad!
			- will drive adoption, owns budget, has vast interest
		- **CEO** / co-founder
			- \"why does everyone keep asking me the same questions.\"
			- May delegate
		- **Senior engineer** / tech lead <-- ==USER and advocate==
			- lives in the pain
			- **bottom-up demand signal**

Extra Factors
- first painful departure -- lost info
- team crossing ~12 ppl
	- work duplication, repeated slack questions
- hiring of highly technical ppl who cannot find anything
- post-incident relief

We talked for about an hour ->
- 30 minutes on her problems
- 30 minutes on my solution

==Unknown Unknowns== -- ppl do not now what to even search for! This is where GraphRAG + agents beat traditional FAQ RAG

She liked cloud inference more than Anthropic / OpenAI APIs
Physical touch builds trust I guess: we have started from a handshake and finished with a hug

## Knowledge
### What is the problem
- They hire new ppl and cannot trust them for a year
- New ppl are extremely **risk-averse** compared to Tess
- **Her Pain:**
	- Tess has 6 hours of meetings daily + cannot do the work she wants for 4 years
		- constant ppl mentorship
		- ppl needing a client call over-sitting in case they fail
		- back-and-force with client's IT team on what they need
	- takes too long to teach business ppl how to use excel task completion prediction excel sheet
- **Attempts**
	- They also have video guides on internal hour-billing, software configuration, culture
	- They also have lots of word documents
	- Tess tried to configure Copilot no code, but failed in giving structured outputs
- **What they have now**
	- They use ==SharePoint== as the source of truth
	- they use the whole microsoft suite: teams, copilot, sharepoint, some random graph DB
	- they have 300 ppl

I can clearly see that there are AI-KMS for enterprise:
- Amazon Q + some wrappers from AWS Marketplace
- Google has their own enterprise KMS
- Zen - YC baked
	- Temporal graph agent memory
	- open sourced part of the product -> Graphiti
"""

mentions = extract_mentions_from_doc(doc)
print(f"{len(mentions)} mentions extracted")

40 mentions extracted


In [ ]:
# Display all mentions grouped by type
by_type: dict[str, list[dict]] = defaultdict(list)
for m in mentions:
    by_type[m["type"]].append(m)

for entity_type, items in sorted(by_type.items()):
    print(f"\n{'─' * 60}")
    print(f"  {entity_type}  ({len(items)} mentions)")
    print(f"{'─' * 60}")
    for m in items:
        print(f"  text    : {m['text']!r}")
        print(f"  context : {m['context'][:100]!r}")
        print()

In [ ]:
# Unique mention texts per type — quick coverage check
print("Unique mention texts per type:\n")
for entity_type, items in sorted(by_type.items()):
    texts = sorted({m["text"] for m in items})
    print(f"  [{entity_type}]")
    for t in texts:
        print(f"    {t}")

## Gleaning — the measurable recall lever

The right way to raise recall is **gleaning**: a second prompt that shows the model its own first-pass output and asks "what did you miss?" Recall goes up through a mechanism you can quantify — new mentions found / total mentions after gleaning — without touching the type definition or injecting relevance language into the extraction prompt.

Why gleaning beats prompt-tuning for recall:
- The type definition stays fixed, so you're measuring lift on the same task
- You can see exactly which sentences the first pass skipped, making errors auditable
- A single extra call per chunk; stop gleaning when the lift drops below a threshold

Each gleaned mention carries a `char_offset`: the character position of its sentence within the original chunk, so you can trace every miss back to its exact location.

In [43]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

GLEANING_SYSTEM_PROMPT = """You are auditing an entity mention extraction for completeness. This is the second pass of Stage 1 in a knowledge-graph pipeline: each mention you emit is later embedded (exact span + surrounding sentence) and resolved into a graph node.

You will receive the original text, followed by the (text, context) pairs already extracted by the first pass.
Your task: emit ONLY mention occurrences the first pass MISSED. A mention occurrence is a specific span in a specific sentence — if a name already in the list occurs AGAIN in a sentence not covered by the list, that new occurrence WAS missed and you must emit it. Do not re-emit any (text, context) pair already listed.

Apply the same extraction rules as the first pass:

IN SCOPE
- Proper names: people, organizations, places, facilities, products, software, systems, events, works, datasets. Numerals inside names are part of the name ("HippoRAG 2").
- Named technical terms that function as the established name of a concept, method, or artifact — not ad-hoc descriptions.
- Definite/demonstrative nominals ("the company", "this model") ONLY if a reader of this text alone can identify the unique entity referred to. Kind-level or generic uses are out.

NEVER IN SCOPE
- Pronouns and bare demonstratives ("it", "they", "this" without a noun head).
- Generic common nouns and bare plurals ("users", "the results", "systems"), unless part of a longer specific term.
- Scalar values: numbers, money, counts, dates, times, percentages, durations. A separate stage handles these.
- Relations or facts.

SPAN AND CONTEXT RULES
- Maximal name span; never a proper-name substring of a longer proper name ("University of Toronto", not also "Toronto"). But DO extract an embedded name making its own referential contribution ("Neo4j" in "the Neo4j database").
- Acronym introduced with its expansion: two mentions sharing one context, never the combined parenthesized span.
- "text": verbatim, character-for-character, findable by exact string search inside its "context".
- "context": the complete sentence, verbatim; the whole line/cell for headings and fragments; every sentence touched if the mention crosses a boundary.

TYPE FIELD
Lowercase singular noun. Prefer: person, organization, location, facility, product, software, method, concept, event, work, dataset, field. Runnable artifact → software; procedure/algorithm → method; abstract non-procedure → concept; paper/report/standard → work. Mint a new type only if no anchor fits, as a single generic noun reusable across unrelated documents.

POSTURE
Common first-pass misses to check for deliberately: repeated occurrences of already-found names; mentions inside headings, lists, tables, captions, and parentheticals; embedded modifier names; acronym or expansion halves where only one was caught; specific-referent nominals; multiword technical terms. Favor recall on names and established terms. Do NOT use this pass to pad the list with generic nouns the first pass correctly excluded — an entity the first pass skipped and an exclusion the first pass applied correctly look similar; re-check against the rules above before emitting. If nothing was missed, return an empty mentions list."""


def extract_mentions_with_model(text: str, model: str = "gpt-4.1") -> list[dict]:
    """Extract entity mentions using any OpenAI model."""
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "entity_mentions",
                "strict": True,
                "schema": MENTION_SCHEMA,
            },
        },
    )
    return json.loads(response.choices[0].message.content)["mentions"]


def extract_mentions_from_doc_with_model(
    doc: str, model: str = "gpt-4.1", max_workers: int = 8
) -> list[dict]:
    """Parallel episode extraction; each mention carries episode_idx and episode text."""
    episodes = chunk_doc(doc)

    def _extract(args: tuple[int, str]) -> tuple[int, list[dict]]:
        episode_idx, episode = args
        mentions = extract_mentions_with_model(episode, model=model)
        for m in mentions:
            m["episode_idx"] = episode_idx
            m["episode"]     = episode
        return episode_idx, mentions

    results: dict[int, list[dict]] = {}
    with ThreadPoolExecutor(max_workers=min(len(episodes), max_workers)) as executor:
        for episode_idx, mentions in executor.map(_extract, enumerate(episodes)):
            results[episode_idx] = mentions

    all_mentions: list[dict] = []
    for idx in sorted(results):
        all_mentions.extend(results[idx])
    return all_mentions


def glean_chunk(
    episode: str,
    first_pass: list[dict],
    model: str = "gpt-4.1",
) -> list[dict]:
    """Re-prompt pass on one episode: returns only net-new mention occurrences."""
    first_pairs = {(m["text"], m["context"]) for m in first_pass}
    already = json.dumps(
        [{"text": t, "context": c} for t, c in sorted(first_pairs)],
        ensure_ascii=False,
    )
    user_msg = (
        f"TEXT:\n{episode}\n\n"
        f"ALREADY EXTRACTED (do not re-emit these text+context pairs):\n{already}\n\n"
        "Which mention occurrences were missed?"
    )
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": GLEANING_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "entity_mentions",
                "strict": True,
                "schema": MENTION_SCHEMA,
            },
        },
    )
    raw = json.loads(response.choices[0].message.content)["mentions"]

    new_mentions = []
    for m in raw:
        if (m["text"], m["context"]) in first_pairs:
            continue
        try:
            m["char_offset"] = episode.index(m["context"])  # first occurrence of context
        except ValueError:
            m["char_offset"] = -1
        m["gleaned"] = True
        new_mentions.append(m)
    return new_mentions

def _process_episode(
    episode_idx: int, episode: str, model: str
) -> tuple[int, list[dict], dict]:
    """First pass + glean for one episode. Thread-safe (no shared state)."""
    first_pass = extract_mentions_with_model(episode, model=model)
    for m in first_pass:
        m["episode_idx"] = episode_idx
        m["episode"]     = episode

    gleaned = glean_chunk(episode, first_pass, model=model)
    for m in gleaned:
        m["episode_idx"] = episode_idx
        m["episode"]     = episode

    stats = {
        "episode_idx":      episode_idx,
        "episode":          episode,
        "episode_preview":  episode[:80].replace("\n", " "),
        "first_pass_count": len(first_pass),
        "gleaned_count":    len(gleaned),
        "gleaned_mentions": gleaned,
    }
    return episode_idx, first_pass + gleaned, stats


def extract_with_gleaning(
    doc: str,
    model: str = "gpt-4.1",
    max_workers: int = 8,
) -> tuple[list[dict], list[dict]]:
    """
    Parallel extraction with gleaning. All episodes run concurrently;
    within each episode first-pass and glean are serial (glean needs first-pass output).
    Returns (all_mentions, per_episode_stats) ordered by episode_idx.
    """
    episodes = chunk_doc(doc)
    results: dict[int, tuple[list[dict], dict]] = {}

    with ThreadPoolExecutor(max_workers=min(len(episodes), max_workers)) as executor:
        futures = {
            executor.submit(_process_episode, idx, ep, model): idx
            for idx, ep in enumerate(episodes)
        }
        for future in as_completed(futures):
            idx, mentions, stats = future.result()
            results[idx] = (mentions, stats)

    all_mentions: list[dict] = []
    all_stats:    list[dict] = []
    for idx in sorted(results):
        mentions, stats = results[idx]
        all_mentions.extend(mentions)
        all_stats.append(stats)

    return all_mentions, all_stats

In [88]:
# ── gleaning demo on the sample doc ───────────────────────────────────────────
gleaned_mentions_demo, glean_stats_demo = extract_with_gleaning(doc, model="gpt-4.1")

total_first   = sum(s["first_pass_count"] for s in glean_stats_demo)
total_gleaned = sum(s["gleaned_count"]    for s in glean_stats_demo)
lift_pct = 100 * total_gleaned / (total_first + total_gleaned) if (total_first + total_gleaned) else 0

print(f"First pass : {total_first} mentions")
print(f"Gleaned    : {total_gleaned} new mentions  ({lift_pct:.1f}% of total)")
print(f"Total      : {total_first + total_gleaned}")

# ── per-type breakdown of gleaned mentions ─────────────────────────────────────
glean_by_type: dict[str, list[dict]] = defaultdict(list)
for s in glean_stats_demo:
    for m in s["gleaned_mentions"]:
        glean_by_type[m["type"]].append(m)

print("\nGleaned by type:")
for t, items in sorted(glean_by_type.items(), key=lambda x: -len(x[1])):
    print(f"  {t:<12}  {len(items)}  {[m['text'] for m in items]}")

First pass : 42 mentions
Gleaned    : 6 new mentions  (12.5% of total)
Total      : 48

Gleaned by type:
  event         2  ['Seed', 'Series A']
  organization  1  ['the company']
  concept       1  ['agents']
  software      1  ['Copilot']
  product       1  ['the product']


## Embedding with all-mpnet-base-v2 (symmetric)

Each mention is embedded by span-pooling only the tokens that overlap with the mention's character offsets within its `context` sentence. This gives a contextualised vector without the asymmetric query/passage distinction that E5 requires.

In [45]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "mixedbread-ai/mxbai-embed-large-v1"
# "sentence-transformers/all-mpnet-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()
print(f"loaded {MODEL}")


def similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors."""
    va, vb = np.array(a), np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))


MAX_TOKENS = 512
CHARS_PER_TOKEN = 4


def embed_span(text: str, span_start: int, span_end: int) -> list[float]:
    """Contextual vector for text[span_start:span_end] via token-level span pooling."""
    if len(text) > MAX_TOKENS * CHARS_PER_TOKEN:
        window = (MAX_TOKENS * CHARS_PER_TOKEN) // 2
        chunk_start = max(0, span_start - window)
        chunk_end = min(len(text), span_end + window)
        text = text[chunk_start:chunk_end]
        span_start -= chunk_start
        span_end -= chunk_start

    encoding = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_TOKENS,
    )
    offset_mapping = encoding.pop("offset_mapping")[0]

    with torch.no_grad():
        output = model(**encoding)

    token_embeddings = output.last_hidden_state[0]
    mask = torch.tensor([
        tok_s < span_end and tok_e > span_start
        for tok_s, tok_e in offset_mapping.tolist()
    ])
    span_tokens = token_embeddings[mask]
    if span_tokens.shape[0] == 0:
        span_tokens = token_embeddings
    return span_tokens.mean(dim=0).numpy().tolist()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

loaded mixedbread-ai/mxbai-embed-large-v1


In [47]:
EMBED_BATCH_SIZE = 64  # sequences per forward pass; increase if you have headroom


def _apply_window(text: str, span_start: int, span_end: int) -> tuple[str, int, int]:
    """Mirror the window-clipping logic from embed_span."""
    if len(text) > MAX_TOKENS * CHARS_PER_TOKEN:
        window      = (MAX_TOKENS * CHARS_PER_TOKEN) // 2
        chunk_start = max(0, span_start - window)
        chunk_end   = min(len(text), span_end + window)
        text        = text[chunk_start:chunk_end]
        span_start -= chunk_start
        span_end   -= chunk_start
    return text, span_start, span_end


def embed_spans_batch(items: list[tuple[str, int, int]]) -> list[list[float]]:
    """One forward pass for a batch of (context, span_start, span_end) tuples."""
    windowed = [_apply_window(ctx, s, e) for ctx, s, e in items]

    encoding = tokenizer(
        [w[0] for w in windowed],
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        padding=True,
        max_length=MAX_TOKENS,
    )
    offset_mappings = encoding.pop("offset_mapping")  # (B, L, 2)

    with torch.no_grad():
        hidden = model(**encoding).last_hidden_state  # (B, L, D)

    results = []
    for i, (_, span_start, span_end) in enumerate(windowed):
        mask = torch.tensor([
            s < span_end and e > span_start
            for s, e in offset_mappings[i].tolist()
        ])
        tokens = hidden[i][mask]
        if tokens.shape[0] == 0:
            tokens = hidden[i]
        results.append(tokens.mean(dim=0).numpy().tolist())
    return results


In [53]:
# Embed each mention using its context sentence as the surrounding text.
# Mentions with the same text across different contexts get their vectors averaged
# so we end up with one vector per unique entity surface form.
vectors: dict[str, list[float]] = {}
mention_types: dict[str, str] = {}
mention_contexts: dict[str, list[str]] = defaultdict(list)

skipped = 0
for m in mentions:
    text, ctx, mtype = m["text"], m["context"], m["type"]
    mention_contexts[text].append(ctx)
    try:
        start = ctx.index(text)
    except ValueError:
        skipped += 1
        continue
    vec = embed_span(ctx, start, start + len(text))
    if text in vectors:
        # average with existing vector (handles multiple contexts)
        vectors[text] = np.mean([vectors[text], vec], axis=0).tolist()
    else:
        vectors[text] = vec
        mention_types[text] = mtype

print(f"{len(vectors)} unique entity surfaces embedded  ({skipped} skipped — text not verbatim in context)")

NameError: name 'mentions' is not defined

## Consensus Correlation Clustering (100 runs)

The pivot algorithm is a randomised 3-approximation — different pivot orderings produce slightly different partitions. Running it 100 times and counting how often each pair of entities lands in the same cluster gives a stable co-occurrence matrix. One final pivot pass on that matrix produces the consensus clustering.

In [55]:
def _build_sim_matrix(vecs: dict[str, list[float]], entities: list[str]) -> np.ndarray:
    """Cosine similarity matrix via L2-normalised dot product. Computed once per clustering call."""
    X = np.array([vecs[e] for e in entities], dtype=np.float32)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm = X / np.clip(norms, 1e-10, None)
    return X_norm @ X_norm.T  # (N, N), float32


def _pivot_run(sim_matrix: np.ndarray, threshold: float) -> list[list[int]]:
    """
    One randomised pivot pass over entity indices.
    Uses precomputed sim_matrix rows — numpy slicing releases the GIL so
    multiple threads can run this concurrently without contention.
    """
    n = sim_matrix.shape[0]
    order = np.random.permutation(n)
    assigned = np.zeros(n, dtype=bool)
    clusters: list[list[int]] = []
    for pivot in order:
        if assigned[pivot]:
            continue
        assigned[pivot] = True
        members = np.where((sim_matrix[pivot] > threshold) & ~assigned)[0]
        assigned[members] = True
        clusters.append([pivot] + members.tolist())
    return clusters


def correlation_clustering(
    vecs: dict[str, list[float]], threshold: float = 0.6
) -> list[list[str]]:
    """Vectorized pivot algorithm: builds sim matrix once, single pass."""
    entities   = list(vecs.keys())
    sim_matrix = _build_sim_matrix(vecs, entities)
    return [[entities[i] for i in c] for c in _pivot_run(sim_matrix, threshold)]


def consensus_clustering(
    vecs: dict[str, list[float]],
    threshold: float = 0.6,
    n_runs: int = 100,
    max_workers: int = 8,
) -> list[list[str]]:
    """
    Parallel consensus clustering.
    - Similarity matrix built once (O(N²·D) where D=embedding dim).
    - n_runs pivot passes run in ThreadPoolExecutor; numpy ops release the GIL.
    - Each run contributes a local co-occurrence matrix summed in the main thread.
    - Final pivot runs directly on the normalised co-occurrence matrix — O(N²),
      no re-embedding or N³ matmul. Works in Jupyter notebooks (threads, not processes).
    """
    entities   = list(vecs.keys())
    n          = len(entities)
    sim_matrix = _build_sim_matrix(vecs, entities)

    def _one_run(_: int) -> np.ndarray:
        local_co = np.zeros((n, n), dtype=np.float32)
        for cluster in _pivot_run(sim_matrix, threshold):
            if len(cluster) < 2:
                continue
            idx = np.array(cluster, dtype=np.intp)
            # mark all pairs in this cluster as co-occurring (diagonal included —
            # self-similarity never triggers assignment because assigned[pivot]=True
            # is set before members are queried)
            local_co[np.ix_(idx, idx)] += 1.0
        return local_co

    co_occurrence = np.zeros((n, n), dtype=np.float64)
    with ThreadPoolExecutor(max_workers=min(n_runs, max_workers)) as executor:
        for local_co in executor.map(_one_run, range(n_runs)):
            co_occurrence += local_co
    co_occurrence /= n_runs  # normalise to [0, 1]: fraction of runs each pair co-occurred

    # Final pivot on the co-occurrence matrix directly — O(N²), skips the O(N³) matmul
    co_float32     = co_occurrence.astype(np.float32)
    final_clusters = _pivot_run(co_float32, threshold=0.5)
    return [[entities[i] for i in cluster] for cluster in final_clusters]


In [57]:
# Tune threshold here — lower = fewer, broader clusters; higher = more, tighter clusters
THRESHOLD = 0.70
N_RUNS = 100

clusters = consensus_clustering(vectors, threshold=THRESHOLD, n_runs=N_RUNS)

# sort clusters largest-first for display
clusters.sort(key=len, reverse=True)
print(f"{len(clusters)} clusters from {len(vectors)} entities  (threshold={THRESHOLD}, runs={N_RUNS})\n")
for i, c in enumerate(clusters):
    print(f"  [{i+1:>2}] ({len(c)})  {c}")

AxisError: axis 1 is out of bounds for array of dimension 1

## Cluster visualisation

Each card = one cluster. Members are shown with their type badge and the first context sentence they appeared in. Singletons (clusters of 1) are collapsed into a separate section at the bottom.

## HDBSCAN Clustering

Density-based hierarchical clustering — no threshold to tune, just `min_cluster_size`. Entities that don't fit any dense region are marked as **noise** (label `-1`) rather than forced into a cluster. Good complement to correlation clustering: CC has no noise concept and every entity lands somewhere.

In [59]:
from sklearn.cluster import HDBSCAN

MIN_CLUSTER_SIZE = 2
MIN_SAMPLES      = 1


def hdbscan_clustering(
    vecs: dict[str, list[float]],
    min_cluster_size: int = 2,
    min_samples: int = 1,
    metric: str = "cosine",
    n_jobs: int = -1,
) -> tuple[list[list[str]], list[str]]:
    """
    HDBSCAN clustering. n_jobs=-1 uses all CPU cores via joblib (notebook-safe).
    Returns (clusters, noise).
    """
    entities = list(vecs.keys())
    X        = np.array([vecs[e] for e in entities])

    labels = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        n_jobs=n_jobs,
    ).fit_predict(X)

    cluster_map: dict[int, list[str]] = defaultdict(list)
    noise: list[str] = []
    for entity, label in zip(entities, labels):
        if label == -1:
            noise.append(entity)
        else:
            cluster_map[label].append(entity)

    return sorted(cluster_map.values(), key=len, reverse=True), noise


hdb_clusters, hdb_noise = hdbscan_clustering(
    vectors, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES
)

print(
    f"{len(hdb_clusters)} clusters, {len(hdb_noise)} noise points"
    f"  (min_cluster_size={MIN_CLUSTER_SIZE})\n"
)
for i, c in enumerate(hdb_clusters):
    print(f"  [{i+1:>2}] ({len(c)})  {c}")
if hdb_noise:
    print(f"\n  noise: {hdb_noise}")

/Users/igor/Desktop/knowledgeGraphs/core/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [106]:
[ 1] (6)  ['SharePoint', 'microsoft suite', 'teams', 'copilot', 'sharepoint', 'graph DB']
  [ 2] (5)  ["CTO's", "COO's", 'late Seed', 'Series A', 'B2B SaaS startups']
  [ 3] (5)  ['GraphRAG', 'agents', 'FAQ RAG', 'Temporal graph agent memory', 'Graphiti']
  [ 4] (4)  ['video guides', 'internal hour-billing', 'software configuration', 'culture']
  [ 5] (3)  ['Fintech', 'biotech', 'B2B SaaS']
  [ 6] (3)  ['cloud inference', 'Anthropic', 'OpenAI APIs']
  [ 7] (2)  ['Breaking point for Buyers of AI KMS', 'Potential Markets']
  [ 8] (2)  ['CTO', 'VP of Engineering']
  [ 9] (2)  ['COO', 'Head of Ops']
  [10] (2)  ['company', 'founders']
  [11] (2)  ['onboarding', 'tribal knowledge']
  [12] (2)  ['CEO', 'co-founder']
  [13] (2)  ['Senior engineer', 'tech lead']
  [14] (2)  ['Tess', 'Copilot no code']
  [15] (2)  ['business ppl', 'excel task completion prediction excel sheet']
  [16] (2)  ['AI-KMS', 'enterprise']
  [17] (2)  ['Amazon Q', 'AWS Marketplace']
  [18] (2)  ['Google', 'enterprise KMS']
  [19] (2)  ['Zen', 'YC']

  noise: ['Late Seed / Series A startups', 'seed', 'Champions + Buyers', 'KM', 'tools budget', 'stakeholders', 'team', 'slack', "client's IT team", 'word documents']



  [ 1] (6)  ['SharePoint', 'microsoft suite', 'teams', 'copilot', 'sharepoint', 'graph DB']
  [ 2] (5)  ["client's IT team", 'video guides', 'internal hour-billing', 'software configuration', 'culture']
  [ 3] (4)  ['GraphRAG', 'agents', 'FAQ RAG', 'Temporal graph agent memory']
  [ 4] (3)  ['Fintech', 'biotech', 'B2B SaaS']
  [ 5] (3)  ['cloud inference', 'Anthropic', 'OpenAI APIs']
  [ 6] (2)  ['Breaking point for Buyers of AI KMS', 'Potential Markets']
  [ 7] (2)  ["CTO's", "COO's"]
  [ 8] (2)  ['Series A', 'B2B SaaS startups']
  [ 9] (2)  ['CTO', 'VP of Engineering']
  [10] (2)  ['COO', 'Head of Ops']
  [11] (2)  ['company', 'founders']
  [12] (2)  ['onboarding', 'tribal knowledge']
  [13] (2)  ['CEO', 'co-founder']
  [14] (2)  ['Senior engineer', 'tech lead']
  [15] (2)  ['Tess', 'Copilot no code']
  [16] (2)  ['business ppl', 'excel task completion prediction excel sheet']
  [17] (2)  ['AI-KMS', 'enterprise']
  [18] (2)  ['Amazon Q', 'AWS Marketplace']
  [19] (2)  ['Google', 'enterprise KMS']
  [20] (2)  ['Zen', 'YC']

  noise: ['late Seed', 'Late Seed / Series A startups', 'seed', 'Champions + Buyers', 'KM', 'tools budget', 'stakeholders', 'team', 'slack', 'word documents', 'Graphiti']

IndentationError: unexpected indent (1942149046.py, line 2)

---

## Bulk extraction from `/Applications/reink`

### Model selection

| Model | Input $/1M | Output $/1M | Context | Notes |
|---|---|---|---|---|
| `gpt-4.1` | $2.00 | $8.00 | 1M | Highest recall; best at subtle/technical entities |
| `gpt-4.1-mini` | $0.40 | $1.60 | 1M | ~5× cheaper than 4.1; same architecture family, very close quality |
| `gpt-4o-mini` | $0.15 | $0.60 | 128K | ~13× cheaper than 4.1; weaker on edge-case and implicit entities |

**Cost for this 50k-token run (input + ~10k output):**

| Model | Input | Output | Total |
|---|---|---|---|
| `gpt-4.1` | $0.10 | $0.08 | **~$0.18** |
| `gpt-4.1-mini` | $0.02 | $0.016 | **~$0.04** |
| `gpt-4o-mini` | $0.0075 | $0.006 | **~$0.014** |

**Recommendation: `gpt-4.1-mini`** — 5× cheaper than full `gpt-4.1` with nearly identical recall on structured personal notes. Upgrade to `gpt-4.1` if you notice missed entities on dense technical passages. Change `EXTRACTION_MODEL` in the cell below to swap without touching the rest of the pipeline.

In [ ]:
import pathlib
import tiktoken

NOTES_DIR = pathlib.Path("/Applications/reink")
CACHE_PATH = pathlib.Path("entity_cache_v1.json")
TOKEN_BUDGET = 50_000
EXTRACTION_MODEL = "gpt-4.1-mini"  # change to "gpt-4.1" for max recall


def count_tokens(text: str, model: str = EXTRACTION_MODEL) -> int:
    """Count tokens using the model's tokenizer; fall back to cl100k_base."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))


def load_cache() -> dict:
    if CACHE_PATH.exists():
        with CACHE_PATH.open() as f:
            return json.load(f)
    return {"processed_files": [], "mentions": []}


def save_cache(cache: dict) -> None:
    with CACHE_PATH.open("w") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# ── discover files ─────────────────────────────────────────────────────────────
all_md_files = sorted(NOTES_DIR.rglob("*.md"))
cache = load_cache()
already_processed = set(cache["processed_files"])

pending = [p for p in all_md_files if str(p) not in already_processed]
print(f"Total .md files : {len(all_md_files)}")
print(f"Already cached  : {len(already_processed)}")
print(f"Pending         : {len(pending)}")

# ── select files that fit within TOKEN_BUDGET ──────────────────────────────────
selected: list[tuple[pathlib.Path, str, int]] = []  # (path, text, token_count)
cumulative_tokens = 0

for path in pending:
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        continue
    if not text.strip():
        continue
    n_tokens = count_tokens(text)
    if cumulative_tokens + n_tokens > TOKEN_BUDGET:
        break
    selected.append((path, text, n_tokens))
    cumulative_tokens += n_tokens

print(f"\nSelected {len(selected)} files — {cumulative_tokens:,} input tokens")
for path, _, n in selected:
    print(f"  {n:>6,}  {path.relative_to(NOTES_DIR)}")

In [62]:
# extract_mentions_with_model / extract_with_gleaning defined in the gleaning section above.

# ── config ─────────────────────────────────────────────────────────────────────
ENABLE_GLEANING  = True
FILE_WORKERS     = 4   # files processed in parallel
EPISODE_WORKERS  = 8   # episodes parallelised within each file

# ── per-file worker ────────────────────────────────────────────────────────────
_cache_lock = threading.Lock()


def _process_file(
    path: pathlib.Path, text: str
) -> tuple[pathlib.Path, list[dict], list[dict]]:
    """Extract from one file; all mentions tagged with source_file. Thread-safe."""
    print("trying")
    if ENABLE_GLEANING:
        mentions, glean_stats = extract_with_gleaning(
            text, model=EXTRACTION_MODEL, max_workers=EPISODE_WORKERS
        )
        for s in glean_stats:
            s["source_file"] = str(path)
    else:
        mentions = extract_mentions_from_doc_with_model(
            text, model=EXTRACTION_MODEL, max_workers=EPISODE_WORKERS
        )
        glean_stats = []

    for m in mentions:
        m["source_file"] = str(path)

    return path, mentions, glean_stats


# ── run ────────────────────────────────────────────────────────────────────────
if not selected:
    print("Nothing to process — either budget exhausted or all files already cached.")
else:
    all_glean_stats: list[dict] = []
    completed = 0

    with ThreadPoolExecutor(max_workers=min(len(selected), FILE_WORKERS)) as executor:
        futures = {
            executor.submit(_process_file, path, text): (path, n_tokens)
            for path, text, n_tokens in selected
        }
        for future in as_completed(futures):
            path, n_tokens = futures[future]
            file_path, file_mentions, file_glean_stats = future.result()
            completed += 1

            n_gleaned = sum(s["gleaned_count"] for s in file_glean_stats)
            glean_suffix = f"  (+{n_gleaned} gleaned)" if ENABLE_GLEANING else ""
            print(
                f"[{completed}/{len(selected)}] {file_path.name}"
                f"  ({n_tokens:,} tok)  →  {len(file_mentions)} mentions{glean_suffix}"
            )

            with _cache_lock:
                cache["mentions"].extend(file_mentions)
                cache["processed_files"].append(str(file_path))
                save_cache(cache)

            all_glean_stats.extend(file_glean_stats)

    print(
        f"\nDone. Cache: {len(cache['processed_files'])} files, "
        f"{len(cache['mentions'])} total mentions."
    )

    if ENABLE_GLEANING and all_glean_stats:
        bulk_total_first   = sum(s["first_pass_count"] for s in all_glean_stats)
        bulk_total_gleaned = sum(s["gleaned_count"]    for s in all_glean_stats)

trying



KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
# ── load cached mentions for downstream analysis ───────────────────────────────
cache = load_cache()
bulk_mentions: list[dict] = cache["mentions"]

print(f"Loaded {len(bulk_mentions)} mentions from {len(cache['processed_files'])} files")

# Quick per-type breakdown
bulk_by_type: dict[str, int] = defaultdict(int)
for m in bulk_mentions:
    bulk_by_type[m["type"]] += 1
for t, cnt in sorted(bulk_by_type.items(), key=lambda x: -x[1]):
    print(f"  {t:<12}  {cnt}")

for i in cache["mentions"]:
    print(i["text"])

---

## Embedding + clustering over cached entities

Runs the same embed → consensus CC → HDBSCAN pipeline on all cached mentions from `/Applications/reink`.

In [71]:
# ── load precomputed vectors from embed_cache.py ─────────────────────────────
# Run once:  uv run python embed_cache.py
import pathlib, json
from collections import defaultdict

_npz = np.load("vector_cache.npz", allow_pickle=False)
bulk_vectors       = {e: v.tolist() for e, v in zip(_npz["entities"], _npz["matrix"])}
bulk_mention_types = dict(zip(_npz["entities"], _npz["types"]))

# rebuild mention_contexts from cache for display in render_clusters
_cache = json.loads(pathlib.Path("entity_cache.json").read_text())
bulk_mention_contexts: dict[str, list[str]] = defaultdict(list)
bulk_mention_sources:  dict[str, set[str]]  = defaultdict(set)
for m in _cache["mentions"]:
    bulk_mention_contexts[m["text"]].append(m["context"])
    if m.get("source_file"):
        bulk_mention_sources[m["text"]].add(m["source_file"])

print(f"{len(bulk_vectors)} entity vectors loaded  (dim={len(next(iter(bulk_vectors.values())))})")

1439 entity vectors loaded  (dim=1024)


In [66]:
# ── 1. filter mentions and collect tasks ─────────────────────────────────────
bulk_vectors:          dict[str, list[float]] = {}
bulk_mention_types:    dict[str, str]         = {}
bulk_mention_contexts: dict[str, list[str]]   = defaultdict(list)
bulk_mention_sources:  dict[str, set[str]]    = defaultdict(set)

Task = tuple[str, str, int, int, str]  # (entity_text, episode, start, end, mtype)
tasks: list[Task] = []
skipped_bulk = 0

for m in bulk_mentions:
    text, ctx, mtype = m["text"], m["context"], m["type"]
    episode = m.get("episode", ctx)
    bulk_mention_contexts[text].append(ctx)
    if m.get("source_file"):
        bulk_mention_sources[text].add(m["source_file"])
    try:
        start = episode.index(text)
    except ValueError:
        skipped_bulk += 1
        continue
    tasks.append((text, episode, start, start + len(text), mtype))

# ── 2. parallel batch embed ──────────────────────────────────────────────────
# Limit PyTorch's internal thread pool so it doesn't fight our Python threads.
torch.set_num_threads(1)
N_WORKERS = 6  # tune: roughly half your physical CPU cores

raw_vecs:  dict[str, list[list[float]]] = defaultdict(list)
raw_types: dict[str, str]               = {}
completed = 0
total_tasks = len(tasks)

batches = [tasks[i : i + EMBED_BATCH_SIZE] for i in range(0, total_tasks, EMBED_BATCH_SIZE)]

def _embed_batch(batch: list[Task]) -> list[tuple[str, list[float], str]]:
    spans = [(episode, start, end) for _, episode, start, end, _ in batch]
    vecs  = embed_spans_batch(spans)
    return [(entity_text, vec, mtype) for (entity_text, _, _, _, mtype), vec in zip(batch, vecs)]

# with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
#     futures = {executor.submit(_embed_batch, b): len(b) for b in batches}
#     for future in as_completed(futures):
#         for entity_text, vec, mtype in future.result():
#             raw_vecs[entity_text].append(vec)
#             raw_types[entity_text] = mtype
#         completed += futures[future]
#         if completed % 400 == 0 or completed >= total_tasks:
#             print(f"  embedded {min(completed, total_tasks)}/{total_tasks} …", flush=True)

# # ── 3. average per entity surface ────────────────────────────────────────────
# for text, vecs in raw_vecs.items():
#     bulk_vectors[text]       = np.mean(vecs, axis=0).tolist()
#     bulk_mention_types[text] = raw_types[text]

# print(
#     f"\n{len(bulk_vectors)} unique entity surfaces embedded"
#     f"  ({skipped_bulk} skipped — text not verbatim in episode)"
# )

In [73]:
BULK_THRESHOLD = 0.90
BULK_N_RUNS    = 200

bulk_cc_clusters = consensus_clustering(
    bulk_vectors, threshold=BULK_THRESHOLD, n_runs=BULK_N_RUNS
)
bulk_cc_clusters.sort(key=len, reverse=True)

n_multi     = len([c for c in bulk_cc_clusters if len(c) > 1])
n_singleton = sum(1 for c in bulk_cc_clusters if len(c) == 1)
print(
    f"{len(bulk_cc_clusters)} clusters from {len(bulk_vectors)} entities"
    f"  ({n_multi} multi-member, {n_singleton} singletons)"
    f"  threshold={BULK_THRESHOLD}, runs={BULK_N_RUNS}"
)

633 clusters from 1439 entities  (195 multi-member, 438 singletons)  threshold=0.9, runs=200


In [ ]:
n_multi     = len([c for c in bulk_cc_clusters if len(c) > 1])
n_singleton = sum(1 for c in bulk_cc_clusters if len(c) == 1)
print(f"Consensus CC  —  {n_multi} groups · {n_singleton} singletons · threshold={BULK_THRESHOLD} · {BULK_N_RUNS} runs · {len(bulk_vectors)} entities")
print()

for i, cluster in enumerate(c for c in bulk_cc_clusters if len(c) > 1):
    print(f"  [{i+1:>3}] ({len(cluster)} members)")
    for entity in cluster:
        ctx = bulk_mention_contexts.get(entity, [""])[0]
        ctx_short = ctx[:80].replace("\n", " ").strip()
        print(f"        {entity:<35}  {ctx_short!r}")
    print()


In [75]:
BULK_MIN_CLUSTER_SIZE = 5
BULK_MIN_SAMPLES      = 2

bulk_hdb_clusters, bulk_hdb_noise = hdbscan_clustering(
    bulk_vectors,
    min_cluster_size=BULK_MIN_CLUSTER_SIZE,
    min_samples=BULK_MIN_SAMPLES,
)

n_multi = len([c for c in bulk_hdb_clusters if len(c) > 1])
print(f"HDBSCAN  —  {n_multi} groups · {len(bulk_hdb_noise)} noise · min_cluster_size={BULK_MIN_CLUSTER_SIZE} · {len(bulk_vectors)} entities")
print()

for i, cluster in enumerate(c for c in bulk_hdb_clusters if len(c) > 1):
    print(f"  [{i+1:>3}] ({len(cluster)} members)")
    for entity in cluster:
        ctx = bulk_mention_contexts.get(entity, [""])[0]
        ctx_short = ctx[:80].replace("\n", " ").strip()
        print(f"        {entity:<35}  {ctx_short!r}")
    print()

if bulk_hdb_noise:
    print(f"  noise ({len(bulk_hdb_noise)}): {bulk_hdb_noise}")


/Users/igor/Desktop/knowledgeGraphs/core/.venv/lib/python3.12/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


HDBSCAN  —  87 groups · 148 noise · min_cluster_size=5 · 1439 entities

  [  1] (45 members)
        1 × 2‑hr                             '- **1\xa0×\xa02‑hr** tactics scenario or range day'
        2 × 1‑hr                             '- **2\xa0×\xa01‑hr** tech labs or builds'
        4 × 1‑hr                             '- **4\xa0×\xa01‑hr** detective/\u200bforensics study'
        5 × 2‑hr                             '- **5\xa0×\xa02‑hr** physical blocks (strength\xa0+\xa0martial)'
        Bat‑computer                         '3. **Feedback loops** – Use wearables for body metrics, solve “cases” (CTF, esca'
        Bruce                                '3. **Feedback loops** – Use wearables for body metrics, solve “cases” (CTF, esca'
        CTF                                  '3. **Feedback loops** – Use wearables for body metrics, solve “cases” (CTF, esca'
        Coursera CSI                         '- _Know how_: Coursera CSI, MIT\xa0OpenCourseWare cryptography, Gracie Universit

## LLM Entity Resolution

Within each cluster, ask the model to merge entities that refer to the same real-world thing.
Shows what went in (entity + context) and what the model decided to merge.

In [105]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm as tqdm_nb
from collections import Counter
import json

# ── prompt (your revised RESOLVE_SYSTEM — replace if you've edited it) ────────
RESOLVE_SYSTEM = """You decide which of the given entity mentions denote the SAME real-world entity. These mentions were pre-clustered by embedding similarity, so similarity is NOT evidence of sameness — your job is to find the genuine identity boundaries inside a similar-looking set. The resulting entities become nodes in a knowledge graph traversed by PageRank: a wrong merge fuses two entities' neighborhoods and corrupts every traversal through them; an unmerged duplicate merely splits evidence and is cheap to fix later. Default to SEPARATE.

WHAT COUNTS AS EVIDENCE OF SAMENESS (merge only on these):
- Acronym and its expansion, especially when a context shows them introduced together ("Personalized PageRank (PPR)").
- Morphological variants of one name or term: casing, hyphenation, plural, spelling, punctuation ("Neo4j" / "neo4j"; "co-reference" / "coreference").
- Formal vs. short form of one name with compatible contexts ("Acme Corporation" / "Acme Corp" / "Acme", where contexts describe the same organization).
- An explicit alias statement in a context ("X, also known as Y").

WHAT IS NEVER SUFFICIENT: shared head noun, same type, overlapping topic, one being a substring of the other, or contexts about related subject matter. These are why the mentions were clustered together; they prove nothing.

MANDATORY SEPARATIONS:
- Different specificity levels: versions, editions, sub-units, and qualified variants are DISTINCT entities ("GPT-4" ≠ "GPT-4 Turbo"; "University of Toronto" ≠ "University of Toronto Mississauga") unless a context explicitly equates them.
- Different terms for related concepts: merge only terminological variants of ONE term, never two synonyms or near-synonyms ("entity resolution" ≠ "entity deduplication").
- Same surface form, different referents: contexts describing incompatible facts mean separate groups even for identical strings' variants.
- Generic or anaphoric mentions ("the platform", "the model", "this system"): keep as singletons unless a context from the SAME document explicitly ties one to a named member. Across documents, identical generic phrases denote different things.
- Conflicting types (e.g. person vs. organization): separate, unless the contexts make clear one type label is simply an error.
- Related but distinct things (a company and its product, a method and the paper introducing it): separate.

A mention that matches nothing goes in its own single-member group. This is the expected outcome for most mentions, not a failure.

For each group output:
- canonical: the most complete and specific member name. Never generalize, abbreviate, or invent a name absent from the members.
- members: the input mentions in this group, copied VERBATIM, character-for-character, exactly as given.
- confidence: that ALL members denote one entity. "high" = direct evidence from the list above for every member; "medium" = evidence for the pattern but a member's context is uninformative; "low" = plausible but unverified — prefer splitting such a group into a smaller confident core plus singletons over emitting a low-confidence merge.
- reason: one sentence naming the evidence type used (or the boundary kept).

Every input mention must appear in exactly one group. Do not invent, omit, reword, or duplicate any mention."""

RESOLVE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "groups": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "canonical":  {"type": "string"},
                    "members":    {"type": "array", "items": {"type": "string"}},
                    "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                    "reason":     {"type": "string"},
                },
                "required": ["canonical", "members", "confidence", "reason"],
            },
        }
    },
    "required": ["groups"],
}

# ── optional metadata from bulk_mentions (skipped silently if absent) ─────────
bulk_mention_types:  dict[str, str] | None = None
bulk_mention_counts: dict[str, int] | None = None
if "bulk_mentions" in globals():
    bulk_mention_counts = dict(Counter(m["text"] for m in bulk_mentions))
    bulk_mention_types  = {m["text"]: m["type"] for m in bulk_mentions}


def _pick_contexts(ctxs: list[str], k: int = 2) -> list[str]:
    """Pick up to k contexts spread across the list (proxy for source diversity)."""
    if len(ctxs) <= k:
        return ctxs
    if k == 1:
        return [ctxs[0]]
    step = (len(ctxs) - 1) / (k - 1)
    return [ctxs[round(i * step)] for i in range(k)]


def resolve_cluster(
    entities: list[str],
    contexts: dict[str, list[str]],
    types:  dict[str, str] | None = None,
    counts: dict[str, int] | None = None,
    model: str = "gpt-4.1-mini",
) -> dict:
    """Ask the LLM to group entities in one cluster by real-world identity."""
    lines = []
    for i, e in enumerate(entities, 1):
        picked  = _pick_contexts(contexts.get(e, []), k=2)
        ctx_str = " | ".join(f'"{c[:200].strip()}"' for c in picked) or "(no context)"
        meta = []
        if types  and e in types:  meta.append(types[e])
        if counts and e in counts: meta.append(f"×{counts[e]}")
        tag = f" [{', '.join(meta)}]" if meta else ""
        lines.append(f"{i}. {e}{tag}\n   contexts: {ctx_str}")
    user_msg = (
        "Entity mentions (copy the mention text exactly as written after the number, "
        "without the number):\n" + "\n".join(lines)
    )

    resp = client.with_options(timeout=10.0, max_retries=2).chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": RESOLVE_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "entity_groups", "strict": True, "schema": RESOLVE_SCHEMA},
        },
    )
    result = json.loads(resp.choices[0].message.content)
    return validate_partition(entities, result)


def validate_partition(entities: list[str], result: dict, verbose: bool = True) -> dict:
    """Enforce 'every mention in exactly one group', with normalization fallback
    before dropping a member. Omissions restored as low-confidence singletons."""
    entity_set = set(entities)

    def norm(s: str) -> str:
        return s.strip().strip("'\"").strip()

    norm_map: dict[str, str] = {}
    for e in entities:
        norm_map.setdefault(norm(e), e)

    seen: set[str] = set()
    clean_groups: list[dict] = []
    dropped_examples: list[str] = []
    dropped = 0
    for g in result["groups"]:
        members = []
        for m in g["members"]:
            cand = m if m in entity_set else norm_map.get(norm(m))
            if cand is not None and cand not in seen:
                members.append(cand)
                seen.add(cand)
            else:
                dropped += 1
                if len(dropped_examples) < 5:
                    dropped_examples.append(m)
        if members:
            clean_groups.append({**g, "members": members})

    missing = sorted(entity_set - seen)
    for m in missing:
        clean_groups.append({
            "canonical": m, "members": [m],
            "confidence": "low",
            "reason": "repair: omitted or reworded by model",
        })

    if verbose and (dropped or missing):
        print(f"  ⚠ validate: dropped={dropped} repaired={len(missing)} "
              f"examples={dropped_examples!r}")

    result["groups"]   = clean_groups
    result["repaired"] = len(missing)
    result["dropped"]  = dropped
    return result


# ── pretty print ──────────────────────────────────────────────────────────────
CONF_MARKER = {"high": "✓", "medium": "~", "low": "?"}

def print_resolution(idx: int, entities: list[str], contexts: dict[str, list[str]], result: dict) -> None:
    groups   = result["groups"]
    n_merges = sum(1 for g in groups if len(g["members"]) > 1)
    low_conf = sum(1 for g in groups if g.get("confidence") == "low" and len(g["members"]) > 1)
    repaired = result.get("repaired", 0)
    dropped  = result.get("dropped", 0)
    print(f"\n{'═'*70}")
    print(f"  cluster {idx+1:>3}  ·  {len(entities)} in  →  {len(groups)} out  ·  {n_merges} merge(s)"
          + (f"  ⚠ {low_conf} low-confidence" if low_conf else "")
          + (f"  ⚠ {repaired} repaired" if repaired else "")
          + (f"  ⚠ {dropped} dropped" if dropped else ""))
    print(f"{'═'*70}")

    member_group: dict[str, dict] = {}
    for g in groups:
        for m in g["members"]:
            member_group[m] = g

    printed_groups: set[int] = set()
    for entity in entities:
        ctx    = (contexts.get(entity) or [""])[0][:70].replace("\n", " ").strip()
        group  = member_group.get(entity)
        merged = group is not None and len(group["members"]) > 1
        print(f"  {'⊕' if merged else '·'}  {entity:<32}  {ctx}")

        if group is not None:
            gid = id(group)
            if gid not in printed_groups:
                printed_groups.add(gid)
                conf   = group.get("confidence", "high")
                marker = CONF_MARKER.get(conf, "?")
                if merged:
                    print(f"     ╰─ [{marker} {conf}]  merges {group['members']}")
                print(f"     ╰─ canonical: {group['canonical']!r}  —  {group['reason']}")


# ── run ───────────────────────────────────────────────────────────────────────
RESOLVE_MODEL   = "gpt-4.1-mini"
ESCALATE_MODEL  = "gpt-4.1"
RESOLVE_WORKERS = 8

clusters = bulk_hdb_clusters
## bulk_hdb_clusters, bulk_cc_clusters
multi_clusters = [c for c in clusters if len(c) > 1]
print(f"resolving {len(multi_clusters)} clusters …")

def _all_singleton_fallback(entities: list[str], why: str) -> dict:
    return {
        "groups": [
            {"canonical": e, "members": [e], "confidence": "low", "reason": why}
            for e in entities
        ],
        "repaired": 0, "dropped": 0, "failed": True,
    }

resolved: list[tuple[list[str], dict] | None] = [None] * len(multi_clusters)

with ThreadPoolExecutor(max_workers=RESOLVE_WORKERS) as executor:
    futures = {
        executor.submit(
            resolve_cluster, cluster, bulk_mention_contexts,
            bulk_mention_types, bulk_mention_counts, RESOLVE_MODEL,
        ): i
        for i, cluster in enumerate(multi_clusters)
    }
    for future in tqdm_nb(as_completed(futures), total=len(futures), desc="resolving"):
        i = futures[future]
        try:
            resolved[i] = (multi_clusters[i], future.result())
        except Exception as exc:
            print(f"  ✗ cluster {i} failed: {exc!r}")
            resolved[i] = (multi_clusters[i],
                           _all_singleton_fallback(multi_clusters[i], f"api failure: {exc}"))

# ── escalation: low-confidence merges only (re-add repairs once they're ~0) ───
def _needs_escalation(result: dict) -> bool:
    return any(g["confidence"] == "low" and len(g["members"]) > 1 for g in result["groups"])

escalate_idx = [i for i, (_, r) in enumerate(resolved) if _needs_escalation(r)]
if escalate_idx:
    print(f"escalating {len(escalate_idx)} marginal cluster(s) to {ESCALATE_MODEL} …")
    with ThreadPoolExecutor(max_workers=RESOLVE_WORKERS) as executor:
        futures = {
            executor.submit(
                resolve_cluster, multi_clusters[i], bulk_mention_contexts,
                bulk_mention_types, bulk_mention_counts, ESCALATE_MODEL,
            ): i
            for i in escalate_idx
        }
        for future in tqdm_nb(as_completed(futures), total=len(futures), desc="escalating"):
            i = futures[future]
            try:
                resolved[i] = (multi_clusters[i], future.result())
            except Exception as exc:
                print(f"  ✗ escalation of cluster {i} failed, keeping mini result: {exc!r}")

for i, (entities, result) in enumerate(resolved):
    print_resolution(i, entities, bulk_mention_contexts, result)

# ── mention → entity mapping with confidence ──────────────────────────────────
mention_to_entity: dict[str, dict] = {}
for entities, result in resolved:
    for g in result["groups"]:
        for m in g["members"]:
            mention_to_entity[m] = {
                "canonical":  g["canonical"],
                "confidence": g["confidence"],
                "merged":     len(g["members"]) > 1,
            }
for c in clusters:
    if len(c) == 1 and c[0] not in mention_to_entity:
        mention_to_entity[c[0]] = {"canonical": c[0], "confidence": "high", "merged": False}

# ── final stats ───────────────────────────────────────────────────────────────
n_singletons    = sum(1 for c in clusters if len(c) == 1)
entities_in     = sum(len(entities) for entities, _ in resolved)
groups_out      = sum(len(result["groups"]) for _, result in resolved)
merges          = sum(
    1 for _, result in resolved
    for g in result["groups"] if len(g["members"]) > 1
)
entities_merged = sum(
    len(g["members"]) for _, result in resolved
    for g in result["groups"] if len(g["members"]) > 1
)
total_repaired  = sum(r.get("repaired", 0) for _, r in resolved)
total_dropped   = sum(r.get("dropped", 0)  for _, r in resolved)
n_failed        = sum(1 for _, r in resolved if r.get("failed"))
by_conf = {"high": 0, "medium": 0, "low": 0}
for _, result in resolved:
    for g in result["groups"]:
        if len(g["members"]) > 1:
            by_conf[g.get("confidence", "high")] += 1

raw_surfaces = len(bulk_vectors)
after_cc     = len(clusters)
after_llm    = n_singletons + groups_out

print(f"\n{'━'*50}")
print(f"  ENTITY RESOLUTION PIPELINE")
print(f"{'━'*50}")
print(f"  raw surface forms          : {raw_surfaces:>6}")
print(f"  after CC clustering        : {after_cc:>6}   ({raw_surfaces - after_cc:+d})")
print(f"  after LLM resolution       : {after_llm:>6}   ({after_cc - after_llm:+d})")
print()
print(f"  LLM detail")
print(f"  ├─ multi-clusters in       : {len(multi_clusters):>6}  ({entities_in} entities)")
print(f"  ├─ groups out              : {groups_out:>6}  ({entities_in - groups_out:+d} net)")
print(f"  ├─ escalated clusters      : {len(escalate_idx):>6}")
print(f"  ├─ failed clusters         : {n_failed:>6}")
print(f"  ├─ partition repairs       : {total_repaired:>6}")
print(f"  ├─ hallucinated/dup dropped: {total_dropped:>6}")
print(f"  ├─ merges  high confidence : {by_conf['high']:>6}")
print(f"  ├─ merges medium confidence: {by_conf['medium']:>6}")
print(f"  └─ merges   low confidence : {by_conf['low']:>6}")
print(f"{'━'*50}")

resolving 87 clusters …


resolving:   0%|          | 0/87 [00:00<?, ?it/s]

  ⚠ validate: dropped=1 repaired=0 examples=['Ontario']
  ⚠ validate: dropped=8 repaired=0 examples=['id', 'idxs', 'python', 'python', 'r.weight']
  ⚠ validate: dropped=2 repaired=0 examples=['SF', 'SF']
  ⚠ validate: dropped=6 repaired=6 examples=['central dock [concept, ×1]', 'evening window [concept, ×1]', 'morning window [concept, ×1]', 'sleep [concept, ×3]', 'stability [concept, ×1]']
  ⚠ validate: dropped=2 repaired=2 examples=['Construction [concept, \x7fx1]', 'seeds [concept, \x7fx1]']
  ⚠ validate: dropped=1 repaired=0 examples=['SharePoint']
  ⚠ validate: dropped=1 repaired=0 examples=['FastAPI']
  ⚠ validate: dropped=12 repaired=12 examples=['Earlier science [field, \x7fx1]', 'animal [concept, \x7fx1]', 'body [field, \x7fx1]', 'fulfilment [concept, \x7fx1]', 'intellectual training [concept, \x7fx1]']
  ⚠ validate: dropped=23 repaired=0 examples=['Azure OpenAI', 'Azure OpenAI', 'Data Handling Overview', 'Data Handling Overview', 'Neo4j']
  ⚠ validate: dropped=14 repaired=14 e

escalating:   0%|          | 0/3 [00:00<?, ?it/s]

  ⚠ validate: dropped=11 repaired=11 examples=['Actions [concept, ×1]', 'Energy [field, ×2]', 'Exercise [concept, ×1]', 'Sense of self [concept, ×1]', 'Two forms of mediation [concept, ×1]']

══════════════════════════════════════════════════════════════════════
  cluster   1  ·  45 in  →  45 out  ·  0 merge(s)
══════════════════════════════════════════════════════════════════════
  ·  1 × 2‑hr                          - **1 × 2‑hr** tactics scenario or range day
     ╰─ canonical: '1\xa0×\xa02‑hr'  —  Distinct time duration concepts with different numeric values are separate entities.
  ·  2 × 1‑hr                          - **2 × 1‑hr** tech labs or builds
     ╰─ canonical: '2\xa0×\xa01‑hr'  —  Distinct time duration concepts with different numeric values are separate entities.
  ·  4 × 1‑hr                          - **4 × 1‑hr** detective/​forensics study
     ╰─ canonical: '4\xa0×\xa01‑hr'  —  Distinct time duration concepts with different numeric values are separate entities.
  

## Persist resolved entities → `resolved_entities.json`

Cache gate for the fact-extraction notebook.  
Every entity surface form maps to a **canonical ID** (the resolved canonical name).  
Singletons (never passed to LLM resolution) are included as-is.

In [121]:
import json as _json
from pathlib import Path as _Path

RESOLVED_CACHE = _Path("resolved_entities_v1.json")


def _sources_for(members: list[str]) -> list[str]:
    """Union of source_file(s) across all surface forms merged into one entity."""
    return sorted({s for member in members for s in bulk_mention_sources.get(member, ())})


# ── 1. LLM-resolved multi-member clusters ─────────────────────────────────────
entities_out: list[dict] = []

for cluster_entities, result in resolved:
    for g in result["groups"]:
        entities_out.append({
            "canonical": g["canonical"],
            "members":   g["members"],
            "confidence": g.get("confidence", "high"),
            "reason":    g["reason"],
            "source":    "llm_resolved",
            "source_files": _sources_for(g["members"]),
        })

# ── 2. CC singletons (never went through LLM) ────────────────────────────────
for cluster in bulk_cc_clusters:
    if len(cluster) == 1:
        surface = cluster[0]
        entities_out.append({
            "canonical":  surface,
            "members":    [surface],
            "confidence": "high",
            "reason":     "singleton cluster — no resolution needed",
            "source":     "singleton",
            "source_files": _sources_for([surface]),
        })

# ── 3. Build mention → canonical lookup ───────────────────────────────────────
mention_to_canonical: dict[str, str] = {}
for e in entities_out:
    for member in e["members"]:
        mention_to_canonical[member] = e["canonical"]

# ── 3b. Build canonical → sources lookup (for retrieval) ─────────────────────
canonical_to_sources: dict[str, list[str]] = {
    e["canonical"]: e["source_files"] for e in entities_out
}

# ── 4. Save ───────────────────────────────────────────────────────────────────
payload = {
    "embedding_model": MODEL,
    "resolve_model":   RESOLVE_MODEL,
    "cc_threshold":    BULK_THRESHOLD,
    "total_entities":  len(entities_out),
    "entities":        entities_out,
    "mention_to_canonical": mention_to_canonical,
    "canonical_to_sources": canonical_to_sources,
}

RESOLVED_CACHE.write_text(_json.dumps(payload, ensure_ascii=False, indent=2))

# ── 5. Summary ────────────────────────────────────────────────────────────────
n_llm      = sum(1 for e in entities_out if e["source"] == "llm_resolved")
n_single   = sum(1 for e in entities_out if e["source"] == "singleton")
n_merged   = sum(1 for e in entities_out if len(e["members"]) > 1)
n_surfaces = len(mention_to_canonical)
n_multi_source = sum(1 for e in entities_out if len(e["source_files"]) > 1)

print(f"saved → {RESOLVED_CACHE}")
print(f"  {len(entities_out):>5}  canonical entities")
print(f"  {n_merged:>5}  multi-member (merged)")
print(f"  {n_single:>5}  singletons")
print(f"  {n_surfaces:>5}  surface forms mapped")
print(f"  {n_multi_source:>5}  entities backed by >1 source file")

saved → resolved_entities_v1.json
   1666  canonical entities
     49  multi-member (merged)
    438  singletons
   1402  surface forms mapped
    185  entities backed by >1 source file


## Entity retrieval embeddings (E5 asymmetric)

Each canonical entity now gets **two** embeddings, for two separate jobs:

- **`cluster_embedding`** — the existing `mxbai-embed-large-v1` contextual span embedding
  used above for CC/HDBSCAN clustering, averaged across all surface forms merged into this
  canonical entity. Symmetric use (entities compared against each other), unchanged from
  what's computed earlier in this notebook.
- **`query_embedding`** — a new `intfloat/e5-base-v2` embedding of the canonical name, using
  the asymmetric convention settled on for facts in `relation_extraction.ipynb` Stage 4:
  `"passage: "` prefix here, `"query: "` prefix on the user's question at retrieval time.
  Dot product of the two (vectors are L2-normalized) is what should back query -> entity
  retrieval, e.g. the `n.name_embedding` vector search the retrieval/ package runs at seed time.

Saved to `resolved_entity_vectors.npz`, keyed by canonical name — kept out of
`resolved_entities.json` so the JSON cache doesn't balloon with raw float vectors.

In [110]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from pathlib import Path as _Path

E5_MODEL       = "intfloat/e5-base-v2"
QUERY_PREFIX   = "query: "
PASSAGE_PREFIX = "passage: "
E5_MAX_TOKENS  = 512

torch.set_num_threads(1)
_e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
_e5_model     = AutoModel.from_pretrained(E5_MODEL)
_e5_model.eval()


def _e5_mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool token embeddings over real (non-padding) tokens."""
    mask   = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def embed_e5(
    texts: list[str], is_query: bool = False, batch_size: int = 32
) -> list[list[float]]:
    """Embed with e5-base-v2's asymmetric convention: a prefix is required on every input.

    "query: " for questions, "passage: " for entity names being indexed. Output vectors
    are L2-normalized, so cosine similarity is a plain dot product.
    """
    prefix = QUERY_PREFIX if is_query else PASSAGE_PREFIX
    texts  = [prefix + t for t in texts]

    vectors: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoding = _e5_tokenizer(
            batch, return_tensors="pt", truncation=True,
            padding=True, max_length=E5_MAX_TOKENS,
        )
        with torch.no_grad():
            output = _e5_model(**encoding)
        pooled = _e5_mean_pool(output.last_hidden_state, encoding["attention_mask"])
        normed = F.normalize(pooled, p=2, dim=1)
        vectors.extend(normed.numpy().tolist())
    return vectors


print(f"loaded {E5_MODEL} for entity query-embedding "
      f"(asymmetric: 'query: ' vs 'passage: ' prefixes, both required)")

# ── build both embeddings per canonical entity ────────────────────────────────
ENTITY_VECTOR_CACHE = _Path("resolved_entity_vectors.npz")

canonicals  = [e["canonical"] for e in entities_out]
cluster_dim = len(next(iter(bulk_vectors.values())))

cluster_matrix = np.zeros((len(entities_out), cluster_dim), dtype=np.float32)
for i, e in enumerate(entities_out):
    member_vecs = [bulk_vectors[m] for m in e["members"] if m in bulk_vectors]
    if member_vecs:
        cluster_matrix[i] = np.mean(member_vecs, axis=0)

query_matrix = np.array(embed_e5(canonicals, is_query=False), dtype=np.float32)

np.savez(
    ENTITY_VECTOR_CACHE,
    canonicals=np.array(canonicals),
    cluster_matrix=cluster_matrix,
    query_matrix=query_matrix,
)
print(f"{len(canonicals)} entities  ->  cluster_matrix {cluster_matrix.shape}, "
      f"query_matrix {query_matrix.shape}")
print(f"saved -> {ENTITY_VECTOR_CACHE}")


KeyboardInterrupt



KeyboardInterrupt: 

In [119]:
# for i, (entities, result) in enumerate(resolved):
#     groups = result["groups"]
#     merges = [g for g in groups if len(g["members"]) > 1]
#     if not merges:
#         continue
#     print(f"[{i+1}]  {entities}")
#     for g in merges:
#         print(f"  → {g['canonical']!r}  [{g['confidence']}]  {g['members']}")
#     print()

# That prints only clusters where the LLM actually merged something. If you also want singletons and the full breakdown:

for i, (entities, result) in enumerate(resolved):
    print(f"\n[{i+1}] {entities}")
    for g in result["groups"]:
        marker = "⊕" if len(g["members"]) > 1 else "·"
        print(f"  {marker} {g['canonical']!r}  {g['members']}")

# Or if you just want a flat list of all canonical names after resolution (the final node list):

# all_canonicals = (
#     [c[0] for c in bulk_cc_clusters if len(c) == 1]  # singletons unchanged
#     + [g["canonical"] for _, result in resolved for g in result["groups"]]
# )
# print(f"{len(all_canonicals)} final entity nodes")
# for name in sorted(all_canonicals):
#     print(f"  {name}")


[1] [np.str_('1\xa0×\xa02‑hr'), np.str_('2\xa0×\xa01‑hr'), np.str_('4\xa0×\xa01‑hr'), np.str_('5\xa0×\xa02‑hr'), np.str_('Bat‑computer'), np.str_('Bruce'), np.str_('CTF'), np.str_('Coursera CSI'), np.str_('Daily 20\xa0min'), np.str_('Feedback loops'), np.str_('Gracie University BJJ'), np.str_('MIT\xa0OpenCourseWare'), np.str_('MIT\xa0OpenCourseWare cryptography'), np.str_('Q1 body'), np.str_('Q2 tech'), np.str_('Quarterly skill cycle'), np.str_('Raspberry\xa0Pi'), np.str_('Resource stack'), np.str_('Weekend'), np.str_('Weekly planning'), np.str_('adjustable kettlebells'), np.str_('after‑action reviews'), np.str_('amateur theatre'), np.str_('body metrics'), np.str_('builds'), np.str_('cases'), np.str_('detective KPIs'), np.str_('detective/\u200bforensics study'), np.str_('escape rooms'), np.str_('local makerspace'), np.str_('lock‑pick set'), np.str_('maintenance dosage'), np.str_('martial'), np.str_('martial‑arts gym'), np.str_('meditation/\u200bstoic reflection'), np.str_('physical bl

In [115]:
# ── final stats ───────────────────────────────────────────────────────────────
n_singletons    = sum(1 for c in bulk_cc_clusters if len(c) == 1)
entities_in     = sum(len(entities) for entities, _ in resolved)
groups_out      = sum(len(result["groups"]) for _, result in resolved)
merges          = sum(
    1 for _, result in resolved
    for g in result["groups"] if len(g["members"]) > 1
)
entities_merged = sum(
    len(g["members"]) for _, result in resolved
    for g in result["groups"] if len(g["members"]) > 1
)
by_conf = {"high": 0, "medium": 0, "low": 0}
for _, result in resolved:
    for g in result["groups"]:
        if len(g["members"]) > 1:
            by_conf[g.get("confidence", "high")] += 1

total_nodes_before = len(bulk_cc_clusters)          # clusters = nodes before resolution
total_nodes_after  = n_singletons + groups_out       # singletons unchanged + resolved groups

print(f"\n{'━'*50}")
print(f"  ENTITY RESOLUTION SUMMARY")
print(f"{'━'*50}")
print(f"  clusters before resolution : {total_nodes_before:>6}")
print(f"  ├─ singletons (untouched)  : {n_singletons:>6}")
print(f"  └─ multi-member clusters   : {len(multi_clusters):>6}")
print()
print(f"  entities in multi-clusters : {entities_in:>6}")
print(f"  groups out (after merging) : {groups_out:>6}   ({entities_in - groups_out:+d} collapsed)")
print(f"  merges performed           : {merges:>6}   ({entities_merged} entities merged)")
print(f"  ├─ high confidence         : {by_conf['high']:>6}")
print(f"  ├─ medium confidence       : {by_conf['medium']:>6}")
print(f"  └─ low confidence          : {by_conf['low']:>6}")
print()
print(f"  total nodes before         : {total_nodes_before:>6}")
print(f"  total nodes after          : {total_nodes_after:>6}   ({total_nodes_before - total_nodes_after:+d})")
print(f"{'━'*50}")




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ENTITY RESOLUTION SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  clusters before resolution :    633
  ├─ singletons (untouched)  :    438
  └─ multi-member clusters   :     87

  entities in multi-clusters :   1291
  groups out (after merging) :   1228   (+63 collapsed)
  merges performed           :     49   (112 entities merged)
  ├─ high confidence         :     48
  ├─ medium confidence       :      1
  └─ low confidence          :      0

  total nodes before         :    633
  total nodes after          :   1666   (-1033)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
